# 06 多智能体系统 · 代码示例

**章节**: 06_multi_agent_systems.md
**学习目标**: 模拟 Planner / Tutor / Evaluator 三个 Agent 协作
**A3 关联**: 模拟 A3 系统的虚拟教师团队

In [ ]:
class Blackboard:
    """共享黑板,所有 Agent 通过它通信"""
    def __init__(self):
        self.state = {}
    def write(self, key, value):
        self.state[key] = value
    def read(self, key, default=None):
        return self.state.get(key, default)

class TutorAgent:
    def process(self, bb):
        topic = bb.read("topic")
        bb.write("material", f"[{topic}] 的简化讲解:核心要点 1, 2, 3")
        return bb

class EvaluatorAgent:
    def process(self, bb):
        # 简单判定: 长度超过 10 个字就算 PASSED
        material = bb.read("material", "")
        bb.write("eval_status", "PASSED" if len(material) > 10 else "REJECTED")
        return bb

class PlannerAgent:
    def __init__(self):
        self.tutor = TutorAgent()
        self.evaluator = EvaluatorAgent()
    def run(self, topic):
        bb = Blackboard()
        bb.write("topic", topic)
        for retry in range(3):
            self.tutor.process(bb)
            self.evaluator.process(bb)
            if bb.read("eval_status") == "PASSED":
                return bb.read("material")
        return "FAILED: 达到最大重试次数"

### 解读

- `Blackboard`: 全局共享状态,所有 Agent 读写同一份数据
- `TutorAgent`: 写讲解材料
- `EvaluatorAgent`: 审核材料是否合格
- `PlannerAgent`: 串行调度,失败时重试,达到上限后降级返回

In [ ]:
planner = PlannerAgent()
result = planner.run("Transformer 注意力机制")
print("最终输出:", result)

### 拓展练习

1. 给 Evaluator 加一个「超长」判定,材料字数过多也 REJECTED
2. 增加一个 ProfileAgent,在 Planner 调度前先读取「学生掌握度」
3. 把 Blackboard 改成 dict 列表,记录每次交互历史,方便事后审计